# 05 — Neo4j Load & Showcase Queries

**Goal:** Load the high-risk subgraph into Neo4j AuraDB and run 4 showcase Cypher queries that demonstrate the project's value.

## Why Neo4j?
Parquet files and DataFrames are great for batch analytics. But they're terrible for interactive "what if?" exploration: *"Show me everything that depends on lodash within 3 hops"* is a one-liner in Cypher, but a multi-step Spark job.

Neo4j stores the graph natively — each node and edge is a first-class object with direct pointers to its neighbors. Traversing 3 hops from a node takes milliseconds regardless of how large the overall graph is.

**This is the demo moment**: in an interview or presentation, you open the Neo4j browser and type a package name — the blast radius visualization appears instantly.

## What we load (NOT the full graph)
The Neo4j AuraDB free tier supports ~200k nodes. We load:
- **Top 500 packages by Sentinel Score** + their **2-hop dependency neighborhood**
- All contributors who maintain any of those packages

This is focused enough to be useful and well within the free tier limits.

## Setup: Neo4j AuraDB
1. Go to https://neo4j.com/cloud/platform/aura-graph-database/
2. Create a free AuraDB instance
3. Download the credentials file — it contains your URI, username, and password
4. In Colab, go to **Tools → Secrets** and add:
   - `NEO4J_URI` — e.g. `neo4j+s://xxxxxxxx.databases.neo4j.io`
   - `NEO4J_USER` — usually `neo4j`
   - `NEO4J_PASSWORD` — the generated password from AuraDB

## 0 — Setup

In [ ]:
!pip install -q neo4j pandas pyarrow networkx

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
import json
import pandas as pd
import networkx as nx
from neo4j import GraphDatabase

BASE    = '/content/drive/MyDrive/BlastRadius'
OUT_DIR = f'{BASE}/data/processed'

# Read Neo4j credentials from Colab Secrets (set these up in Tools → Secrets)
NEO4J_URI      = userdata.get('NEO4J_URI')
NEO4J_USER     = userdata.get('NEO4J_USER')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Neo4j AuraDB connection verified.')

In [ ]:
# Load the enriched sentinel scores and graph edges
USE_SAMPLE = True
suffix = '_10k' if USE_SAMPLE else ''
DATA_DIR = f'{BASE}/data/sample' if USE_SAMPLE else f'{BASE}/data/processed'

sentinel_pd = pd.read_parquet(f'{OUT_DIR}/sentinel_enriched.parquet')
edges_pd    = pd.read_parquet(f'{DATA_DIR}/graph_edges{suffix}.parquet')
vertices_pd = pd.read_parquet(f'{DATA_DIR}/graph_vertices{suffix}.parquet')

# Parse cve_ids back to list
sentinel_pd['cve_ids'] = sentinel_pd['cve_ids'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

print(f'Sentinel scores: {len(sentinel_pd):,}')
print(f'Graph edges: {len(edges_pd):,}')

## 1 — Select subgraph to load (top 500 + 2-hop neighborhood)

In [ ]:
# Build NetworkX graph for neighborhood expansion
dep_edges = edges_pd[edges_pd['edge_type'] == 'DEPENDS_ON'][['src', 'dst']]
G = nx.DiGraph()
G.add_edges_from(zip(dep_edges['src'], dep_edges['dst']))

# Top 500 packages by Sentinel Score
top500 = sentinel_pd.nlargest(500, 'sentinel_score')
top500_ids = set('pkg:' + top500['name'].str.lower())

# Expand to 2-hop neighborhood
neighborhood = set(top500_ids)
for pkg_id in top500_ids:
    if pkg_id in G:
        # Predecessors: packages that depend on this one (reverse edges)
        neighborhood |= set(nx.single_source_shortest_path_length(G.reverse(), pkg_id, cutoff=2).keys())
        # Successors: packages this one depends on (forward edges)
        neighborhood |= set(nx.single_source_shortest_path_length(G, pkg_id, cutoff=2).keys())

print(f'Top 500 packages + 2-hop neighborhood: {len(neighborhood):,} package nodes')

# Add contributor nodes that MAINTAIN packages in our subgraph
maintains_edges = edges_pd[edges_pd['edge_type'] == 'MAINTAINS']
relevant_maintains = maintains_edges[maintains_edges['dst'].isin(neighborhood)]
contributor_ids = set(relevant_maintains['src'].unique())

print(f'Contributor nodes to load: {len(contributor_ids):,}')
print(f'Total nodes to load: {len(neighborhood) + len(contributor_ids):,}')

In [ ]:
# Filter edges to only those within our subgraph
all_node_ids = neighborhood | contributor_ids

subgraph_dep_edges = dep_edges[
    dep_edges['src'].isin(all_node_ids) &
    dep_edges['dst'].isin(all_node_ids)
]
subgraph_maint_edges = relevant_maintains

print(f'DEPENDS_ON edges to load: {len(subgraph_dep_edges):,}')
print(f'MAINTAINS edges to load:  {len(subgraph_maint_edges):,}')

## 2 — Neo4j schema setup

**Why constraints?** Neo4j constraints ensure uniqueness and create indexes automatically. Without them, loading 10k nodes one-by-one would do a full scan on every merge — painfully slow. With the constraint, each `MERGE` is an O(1) index lookup.

In [ ]:
def run_query(driver, query, parameters=None):
    with driver.session() as session:
        return list(session.run(query, parameters or {}))

# Clear existing data (safe to re-run this notebook)
print('Clearing existing data...')
run_query(driver, 'MATCH (n) DETACH DELETE n')

# Create uniqueness constraints (also create indexes)
print('Creating constraints...')
run_query(driver, 'CREATE CONSTRAINT pkg_id IF NOT EXISTS FOR (p:Package) REQUIRE p.id IS UNIQUE')
run_query(driver, 'CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:Contributor) REQUIRE u.id IS UNIQUE')

print('Schema ready.')

## 3 — Load Package nodes

**Why batch UNWIND?** Loading nodes one-by-one would mean one round-trip per node. `UNWIND` sends all nodes in one query and lets Neo4j handle the loop internally — roughly 100x faster for bulk loads.

In [ ]:
import math

def load_in_batches(driver, query, records, batch_size=500, label='nodes'):
    """Send records to Neo4j in batches to avoid memory issues."""
    total = len(records)
    n_batches = math.ceil(total / batch_size)
    for i in range(n_batches):
        batch = records[i * batch_size:(i + 1) * batch_size]
        run_query(driver, query, {'batch': batch})
    print(f'Loaded {total:,} {label} in {n_batches} batches')

# Build package node records
# Join neighborhood IDs with sentinel data for properties
sentinel_lookup = sentinel_pd.set_index('name').to_dict('index')
vertices_lookup = vertices_pd.set_index('id').to_dict('index')

pkg_records = []
for pkg_id in neighborhood:
    name = pkg_id.replace('pkg:', '')
    s = sentinel_lookup.get(name, {})
    v = vertices_lookup.get(pkg_id, {})
    pkg_records.append({
        'id':              pkg_id,
        'name':            name,
        'sentinel_score':  float(s.get('sentinel_score', 0.0) or 0.0),
        'pagerank':        float(s.get('pagerank', 0.0) or 0.0),
        'bus_factor':      int(s.get('bus_factor', 1) or 1),
        'risk_tier':       s.get('risk_tier', 'UNKNOWN') or 'UNKNOWN',
        'has_cve':         bool(s.get('has_cve', False)),
        'cve_count':       int(s.get('cve_count', 0) or 0),
        'max_severity':    s.get('max_severity', 'NONE') or 'NONE',
        'cve_ids':         s.get('cve_ids', []) or [],
        'is_critical':     bool(s.get('is_critical', False)),
        'dependent_packages': int(v.get('dependent_packages', 0) or 0),
        'stars':           int(v.get('stars', 0) or 0),
        'is_orphan_risk':  bool(s.get('is_orphan_risk', False))
    })

pkg_load_query = """
UNWIND $batch AS props
MERGE (p:Package {id: props.id})
SET p.name            = props.name,
    p.sentinel_score  = props.sentinel_score,
    p.pagerank        = props.pagerank,
    p.bus_factor      = props.bus_factor,
    p.risk_tier       = props.risk_tier,
    p.has_cve         = props.has_cve,
    p.cve_count       = props.cve_count,
    p.max_severity    = props.max_severity,
    p.cve_ids         = props.cve_ids,
    p.is_critical     = props.is_critical,
    p.dependent_packages = props.dependent_packages,
    p.stars           = props.stars,
    p.is_orphan_risk  = props.is_orphan_risk
"""

load_in_batches(driver, pkg_load_query, pkg_records, label='Package nodes')

In [ ]:
# Load Contributor nodes
contrib_records = [
    {'id': uid, 'login': uid.replace('user:', '')}
    for uid in contributor_ids
]

contrib_load_query = """
UNWIND $batch AS props
MERGE (u:Contributor {id: props.id})
SET u.login = props.login
"""

load_in_batches(driver, contrib_load_query, contrib_records, label='Contributor nodes')

In [ ]:
# Load DEPENDS_ON edges
dep_records = subgraph_dep_edges.to_dict('records')

dep_edge_query = """
UNWIND $batch AS edge
MATCH (a:Package {id: edge.src})
MATCH (b:Package {id: edge.dst})
MERGE (a)-[:DEPENDS_ON]->(b)
"""

load_in_batches(driver, dep_edge_query, dep_records, label='DEPENDS_ON edges')

In [ ]:
# Load MAINTAINS edges
maint_records = subgraph_maint_edges[['src', 'dst', 'total_commits', 'days_since_commit']].to_dict('records')

maint_edge_query = """
UNWIND $batch AS edge
MATCH (u:Contributor {id: edge.src})
MATCH (p:Package {id: edge.dst})
MERGE (u)-[r:MAINTAINS]->(p)
SET r.total_commits    = edge.total_commits,
    r.days_since_commit = edge.days_since_commit
"""

load_in_batches(driver, maint_edge_query, maint_records, label='MAINTAINS edges')

In [ ]:
# Verify load
result = run_query(driver, """
    MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count
    UNION ALL
    MATCH ()-[r]->() RETURN type(r) AS label, count(*) AS count
""")
print('Neo4j graph contents:')
for row in result:
    print(f'  {row["label"]}: {row["count"]:,}')

## 4 — Showcase Queries

These are the 4 queries you'll demo. Each one can also be run directly in the Neo4j Browser at your AuraDB URL.

**How to use the Neo4j Browser:** Go to your AuraDB URL (shown in the AuraDB console), log in, and paste these queries into the query box. The browser renders the result as an interactive graph visualization.

### Query 1: Blast radius — what breaks if package X is compromised?

```cypher
MATCH path = (target:Package {name: 'lodash'})<-[:DEPENDS_ON*1..3]-(dependent:Package)
RETURN path
LIMIT 100
```

In the Browser, this renders as a force-directed graph of everything downstream of lodash within 3 hops.

In [ ]:
# Run Query 1: blast radius of lodash
BLAST_RADIUS_QUERY = """
MATCH (target:Package {name: $pkg_name})<-[:DEPENDS_ON*1..3]-(dependent:Package)
RETURN dependent.name AS dependent, dependent.sentinel_score AS score
ORDER BY score DESC
LIMIT 50
"""

q1_results = run_query(driver, BLAST_RADIUS_QUERY, {'pkg_name': 'lodash'})
print(f'Query 1: Blast radius of lodash ({len(q1_results)} dependents within 3 hops):')
for row in q1_results[:15]:
    print(f"  {row['dependent']:<30} sentinel={row['score']:.4f}")

### Query 2: The fragile backbone — high PageRank + Bus Factor = 1

```cypher
MATCH (p:Package)
WHERE p.bus_factor = 1 AND p.pagerank > 0.01
RETURN p.name, p.pagerank, p.bus_factor, p.risk_tier
ORDER BY p.pagerank DESC
LIMIT 20
```

This is the classic "critical + fragile" pattern — important packages maintained by a single person.

In [ ]:
FRAGILE_QUERY = """
MATCH (p:Package)
WHERE p.bus_factor = 1 AND p.pagerank > 0.01
RETURN p.name AS name, p.pagerank AS pagerank,
       p.bus_factor AS bus_factor, p.risk_tier AS risk_tier,
       p.has_cve AS has_cve
ORDER BY p.pagerank DESC
LIMIT 20
"""

q2_results = run_query(driver, FRAGILE_QUERY)
print(f'Query 2: High-PageRank + Bus Factor = 1 ({len(q2_results)} packages):')
print(f'{"Package":<30} {"PageRank":<12} {"Bus Factor":<12} {"Risk Tier":<12} {"CVE"}')
print('-' * 75)
for row in q2_results:
    print(f"{row['name']:<30} {row['pagerank']:<12.4f} {row['bus_factor']:<12} {row['risk_tier']:<12} {row['has_cve']}")

### Query 3: CVE propagation — who is downstream of a vulnerable package?

```cypher
MATCH (vuln:Package {has_cve: true})<-[:DEPENDS_ON*1..3]-(exposed:Package)
WHERE vuln.max_severity IN ['CRITICAL', 'HIGH']
WITH vuln, count(DISTINCT exposed) AS exposure_count
RETURN vuln.name, vuln.max_severity, vuln.cve_count, exposure_count
ORDER BY exposure_count DESC
LIMIT 15
```

In [ ]:
CVE_PROPAGATION_QUERY = """
MATCH (vuln:Package {has_cve: true})<-[:DEPENDS_ON*1..3]-(exposed:Package)
WHERE vuln.max_severity IN ['CRITICAL', 'HIGH']
WITH vuln, count(DISTINCT exposed) AS exposure_count
RETURN vuln.name AS vuln_package, vuln.max_severity AS severity,
       vuln.cve_count AS cve_count, exposure_count
ORDER BY exposure_count DESC
LIMIT 15
"""

q3_results = run_query(driver, CVE_PROPAGATION_QUERY)
print(f'Query 3: CVE propagation — most exposed downstream packages:')
print(f'{"Package":<25} {"Severity":<12} {"CVEs":<8} {"Exposed Downstream"}')
print('-' * 60)
for row in q3_results:
    print(f"{row['vuln_package']:<25} {row['severity']:<12} {row['cve_count']:<8} {row['exposure_count']}")

### Query 4: Orphan Risk — ghost-maintained packages with many dependents

```cypher
MATCH (p:Package {is_orphan_risk: true})<-[:DEPENDS_ON]-(dep:Package)
WITH p, count(dep) AS direct_dependents
WHERE direct_dependents > 100
RETURN p.name, direct_dependents, p.sentinel_score
ORDER BY direct_dependents DESC
LIMIT 20
```

In [ ]:
ORPHAN_RISK_QUERY = """
MATCH (p:Package {is_orphan_risk: true})<-[:DEPENDS_ON]-(dep:Package)
WITH p, count(dep) AS direct_dependents
RETURN p.name AS name, direct_dependents,
       p.sentinel_score AS sentinel_score,
       p.has_cve AS has_cve
ORDER BY direct_dependents DESC
LIMIT 20
"""

q4_results = run_query(driver, ORPHAN_RISK_QUERY)
print(f'Query 4: Orphan Risk packages with many direct dependents:')
print(f'{"Package":<30} {"Direct Dependents":<20} {"Sentinel Score":<16} {"CVE"}')
print('-' * 75)
for row in q4_results:
    print(f"{row['name']:<30} {row['direct_dependents']:<20} {row['sentinel_score']:<16.4f} {row['has_cve']}")

In [ ]:
driver.close()
print('Neo4j load complete.')
print()
print('To run these queries interactively:')
print(f'  1. Open your AuraDB URL in a browser')
print(f'  2. Log in with your credentials')
print(f'  3. Paste any of the Cypher queries from the cells above')
print(f'  4. The Browser renders results as an interactive graph')
print()
print('Next: open 06_report.ipynb')